# Predicción del caudal medio diario en cuencas de Colombia con una LSTM regional

**Proyecto final — Análisis con Deep Learning, Universidad de los Andes**
Santiago Palacios Loaiza · Diego Leonardo Barón Espitia · Andrés Felipe Mosquera Hernández · Juan Sebastián Rodríguez Avella

**Parte 1: preprocesamiento.** Este notebook lee las series y los atributos de CAMELS-COL, los limpia, los consolida en una tabla y la convierte al formato de NeuralHydrology. Todo se ejecuta aquí, paso a paso, y el resultado es una base nueva en `data/processed/`, lista para entrenar el modelo. El modelado (parte 2) está en `proyecto-final-modelado.ipynb`, que parte de esa base.

> **Regla del proyecto:** el modelo **no** recibe caudal observado como entrada (ni `Q(t-1)` ni nada derivado del caudal).
> Es simulación a partir del clima, y eso es lo que permite usarlo en cuencas sin estación de aforo.

## 0. Entorno

In [1]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Lectura y definicon de rutas de fuentes y base de salida.
ROOT = Path.cwd()
DIR_DATOS = ROOT / "data" / "datos-originales"                  # CSV de atributos originales de CAMELS-COL
DIR_SERIES = DIR_DATOS / "3_Hydrometeorological_data_filtrado"  # series diarias pre-filtradas (1.2)
DIR_SALIDA = ROOT / "data" / "processed"                        # base nueva, lista para el modelo
assert DIR_SERIES.exists(), "Ejecuta el notebook desde la carpeta raíz del proyecto"

# Calendario diario común y partición temporal (justificación en 1.9)
CALENDARIO = pd.date_range("1981-01-01", "2022-12-31", freq="D")
PERIODOS = {"train": ("1981-01-01", "2010-12-31"),
            "validation": ("2011-01-01", "2015-12-31"),
            "test": ("2016-01-01", "2022-12-31")}
EN_TRAIN = (CALENDARIO >= PERIODOS["train"][0]) & (CALENDARIO <= PERIODOS["train"][1])

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

## 1. Preprocesamiento

Se leen las fuentes (1.3), se aplica cada ajuste a las 248 cuencas y al final se escribe la base nueva en `data/processed/` (1.11 y 1.12), lista para entrenar el modelo. Cada subsección describe un problema de los datos, la decisión que se tomó y el código que la aplica. 

### 1.1 Variables del modelo

El modelo simula el caudal diario (`streamflow`, en mm/día) a partir de **4 forzantes dinámicas** (variables indispensables para predecir caudal), que lee para cada día de una ventana de 365 días, y de **48 atributos estáticos**, que describen cada cuenca y no cambian en el tiempo. Nunca recibe caudal observado como entrada.

Del dataset se excluyen:
- **Firmas hidrológicas** (`q_mean`, `baseflow_index`, …): se calculan con el caudal observado. Serían fuga de información y no existen en cuencas no aforadas; se guardan solo para analizar resultados.
- **Metadatos de la estación** (fechas, departamento, coordenadas): describen la estación, no la cuenca. Con las coordenadas, el modelo podría reconocer cada cuenca por su posición en vez de aprender de sus propiedades.
- **Columnas redundantes:** `gauge_elev` (repite `minimum_ele`), `perimeter` (se deduce del área y del índice de Gravelius), 3 de las 4 fórmulas de tiempo de concentración (ordenan igual a las cuencas; se conserva `tc_Johnstone`), `volcaniclastic_rock_perc` (copia de `volcanic_rock_perc`), `Urban_area` (misma clase que `urban_perc`, medida con otro mapa) y `No_classified` (equivale a los cuerpos de agua).

In [2]:
# Forzantes dinámicas: columna en los archivos -> nombre en el proyecto
FORZANTES = {"Precipitacion": "pr", "ETP_": "poten_evapo",
             "Temperatura_minima": "t_min", "Temperatura_maxima": "t_max"}
OBJETIVO = "streamflow"   # caudal en mm/día (1.6)

# Atributos estáticos candidatos. log_area se calcula en 1.7 y los climáticos en 1.8;
# el resto se toma de los CSV de CAMELS-COL. En 1.7 se descartan los casi vacíos.
ATRIBUTOS_CANDIDATOS = {
    "fisiograficos": ["log_area", "gravelius_index", "factor_form", "maximum_ele", "mean_ele", "minimum_ele",
                      "streng_chanel", "equi_slope", "cn_catchment",
                      "tc_Johnstone"],   # reemplaza a tc_representative (1.7)
    "climaticos": ["p_mean", "pet_mean", "aridity", "high_prec_freq", "high_prec_dur",
                   "low_prec_freq", "low_prec_dur"],
    "cobertura": ["forest_perc", "nat_non_forest_form_perc", "agricul_livestock_perc", "non_vegeted_perc",
                  "water_bodies_perc_cob", "non_identi_land_perc"],
    "geologia": ["unconso_depo_perc", "hypabyssal_rock_perc", "metamor_rock_perc", "plutonic_rock_perc",
                 "sedimen_rock_perc", "volcanic_rock_perc"],   # volcaniclastic_rock_perc es copia de volcanic
    "suelos": ["inceptisols_perc", "entisols_perc", "andisols_perc", "alfisols_perc", "vertisols_perc",
               "ultisols_perc", "oxisols_perc", "mollisols_perc", "aridisols_perc", "histosols_perc",
               "urban_perc", "eroded_misce_perc", "rocky_misce_perc", "water_bodies_perc_suelo"],
    "capacidad_uso": [f"class_{i}" for i in range(1, 9)],
}

# Se guardan en la tabla de atributos, pero no entran al modelo
AUXILIARES = ["region", "area", "gauge_lat", "gauge_lon", "gauge_department", "gauge_star", "gauge_end", "gauge_missing"]
FIRMAS_HIDROLOGICAS = ["q_mean", "runoff_ratio", "stream_elas", "slope_fdc", "baseflow_index", "Q5", "Q95",
                       "high_q_freq", "high_q_dur", "low_q_freq", "low_q_dur"]

print(f"{len(FORZANTES)} forzantes y {sum(map(len, ATRIBUTOS_CANDIDATOS.values()))} atributos candidatos:",
      {g: len(cols) for g, cols in ATRIBUTOS_CANDIDATOS.items()})

4 forzantes y 51 atributos candidatos: {'fisiograficos': 10, 'climaticos': 7, 'cobertura': 6, 'geologia': 6, 'suelos': 14, 'capacidad_uso': 8}


### 1.2 Pre-filtrado de las variables dinámicas

Las series diarias (caudal y forzantes) **no son las originales del repositorio Zenodo**. Antes, se aplicó un control de calidad propio que dejó 248 de las 346 cuencas y corrigió la evapotranspiración. El resultado está en `data/datos-originales/3_Hydrometeorological_data_filtrado/`, junto con `cleaning_report.csv`, que registra qué se hizo con cada cuenca. Mientras el código del filtrado no esté en este notebook, el procesamiento parte de esas series ya filtradas.

> ✏️ **Por completar: explicación del pre-filtrado** (Santi)
>
> Describir a grandes rasgos los pasos seguidos. Puntos sugeridos:
> 1. Qué detecto y con qué criterio se aplicaron los cambios (tabla de abajo).
> 2. Cómo se detectaron los picos (`spike`) y los valores congelados (`frozen`) del caudal.
> 3. Cómo y por qué se recalculó la evapotranspiración potencial.
> 4. El código del filtrado, en la celda de abajo, para que el notebook lo ejecute y el resultado se pueda reproducir desde los datos de Zenodo.

| Bandera | Cuencas descartadas con esta bandera | Qué detecta y con qué criterio |
|---|---|---|
| `f_q_gt_p` | 55 | *por completar* |
| `f_regime_mismatch` | 20 | *por completar* |
| `f_flatline` | 19 | *por completar* |
| `f_nested_inconsistent` | 13 | *por completar* |
| `f_delineation` | 6 | *por completar* |
| `f_twin_mismatch` | 5 | *por completar* |

In [3]:
# ✏️ Por completar: código del pre-filtrado. Debe leer las series originales de Zenodo
# y escribir las series filtradas y cleaning_report.csv en DIR_SERIES.

In [4]:
print(DIR_SERIES / "cleaning_report.csv")

C:\Users\Diego\Documents\ADL\proyecto-final\data\datos-originales\3_Hydrometeorological_data_filtrado\cleaning_report.csv


In [5]:
# lee el reporte de limpieza y muestra un resumen
reporte = pd.read_csv(DIR_SERIES / "cleaning_report.csv", dtype={"gauge_id": str})
conservadas = reporte[reporte["kept"]]
print(f"Cuencas en CAMELS-COL: {len(reporte)} | conservadas: {len(conservadas)} | descartadas: {len(reporte) - len(conservadas)}")
print(f"Días de caudal anulados en las conservadas: {int(conservadas['spike_days'].sum())} picos, {int(conservadas['frozen_days'].sum())} congelados")
print(f"Evapotranspiración media: {conservadas['pet_mean_before'].mean():.1f} → {conservadas['pet_mean_after'].mean():.1f} mm/día")
print()
pd.crosstab(reporte["region"], reporte["kept"].map({True: "conservadas", False: "descartadas"}),
            margins=True, margins_name="Total")

Cuencas en CAMELS-COL: 346 | conservadas: 248 | descartadas: 98
Días de caudal anulados en las conservadas: 399 picos, 2767 congelados
Evapotranspiración media: 8.9 → 3.3 mm/día



kept,conservadas,descartadas,Total
region,,,
Amazon,11,11,22
Caribbean,44,9,53
Magdalena-Cauca,136,46,182
Orinoco,37,20,57
Pacific,20,12,32
Total,248,98,346


### 1.3 Lectura de las fuentes

Se leen dos fuentes: las series diarias pre-filtradas (1.2), un archivo por cuenca, y los CSV de atributos originales de CAMELS-COL (archivos 02 y 05 a 11). El archivo 08, de índices climáticos, no se usa porque los índices se recalculan (1.8).

- **Series.** (variables dinámicas) Cada serie se lleva a un calendario diario continuo de 1981 a 2022. Los días que no están en el archivo quedan como filas vacías con estado `sin_registro`; así cada ventana de 365 filas corresponde a 365 días consecutivos.
- **Atributos.** Los CSV tienen inconsistencias de formato que, sin tratamiento, producen errores silenciosos: codificación Latin-1 (02), espacios al final de los nombres de columna (10), 16 384 filas de relleno e identificadores con decimales (11) y la columna `water_bodies_perc` repetida en cobertura y suelos (06 y 07). `leer_csv` las corrige, y `cargar_atributos` valida que no haya identificadores ni columnas repetidas y que todas las cuencas tengan atributos.

In [6]:
def leer_serie(ruta):
    """Lee la serie de una cuenca y la reindexa al calendario diario continuo."""
    df = pd.read_csv(ruta, sep="\t")
    df.index = pd.to_datetime(df.pop("Fecha"), format="%d/%m/%Y")
    if df.index.duplicated().any():
        raise ValueError(f"Fechas repetidas en {ruta.name}")
    df = df.rename(columns={**FORZANTES, "Caudal": "streamflow_m3s", "Caudal_estado": "streamflow_estado"})
    df = df.reindex(CALENDARIO).rename_axis("date")
    df["streamflow_estado"] = df["streamflow_estado"].fillna("sin_registro")
    return df


archivos = sorted(DIR_SERIES.glob("Hydromet_data_*.txt"))
series = {a.stem.removeprefix("Hydromet_data_"): leer_serie(a) for a in archivos}
ids = list(series)
print(f"Series: {len(series)} cuencas x {len(CALENDARIO):,} días")

Series: 248 cuencas x 15,340 días


In [7]:
# Archivo de atributos -> columnas a renombrar (water_bodies_perc está en dos archivos)
ARCHIVOS_ATRIBUTOS = {
    "02_CAMELS_COL_Catchment_information.csv": {},
    "05_CAMELS_COL_Geologic_characteristics.csv": {},
    "06_CAMELS_COL_Land_cover_characteristics.csv": {"water_bodies_perc": "water_bodies_perc_cob"},
    "07_CAMELS_COL_Soil_characteristics.csv": {"water_bodies_perc": "water_bodies_perc_suelo"},
    "09_CAMELS_COL_Hydrological_signatures.csv": {},
    "10_CAMELS_COL_Physiograpic_characteristics.csv": {},
    "11_CAMELS_COL_Land_use_capability.csv": {},
}


def leer_csv(ruta):
    """Lee un CSV de atributos corrigiendo las inconsistencias de formato del dataset."""
    crudo = ruta.read_bytes()
    try:
        texto = crudo.decode("utf-8")
    except UnicodeDecodeError:
        texto = crudo.decode("latin-1")
    encabezado = texto.splitlines()[0]
    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    df = pd.read_csv(io.StringIO(texto), sep=sep, dtype={"gauge_id": str})
    df.columns = df.columns.str.strip()
    df = df.dropna(subset=["gauge_id"])                                                  # filas de relleno
    df["gauge_id"] = df["gauge_id"].str.strip().str.replace(r"\.0+$", "", regex=True)   # "11017010.00"
    if df["gauge_id"].duplicated().any():
        raise ValueError(f"gauge_id repetidos en {ruta.name}")
    return df.set_index("gauge_id")


def cargar_atributos(ids):
    """Une los archivos de atributos en una tabla con una fila por cuenca."""
    attrs = pd.concat([leer_csv(DIR_DATOS / nombre).rename(columns=renombres)
                       for nombre, renombres in ARCHIVOS_ATRIBUTOS.items()], axis=1)
    repetidas = attrs.columns[attrs.columns.duplicated()].tolist()
    if repetidas:
        raise ValueError(f"Columnas con el mismo nombre en varios archivos: {repetidas}")
    faltantes = sorted(set(ids) - set(attrs.index))
    if faltantes:
        raise ValueError(f"Cuencas sin atributos: {faltantes}")

    attrs = attrs.loc[ids].copy()
    reporte = pd.read_csv(DIR_SERIES / "cleaning_report.csv", dtype={"gauge_id": str}).set_index("gauge_id")
    attrs["region"] = reporte["region"].reindex(attrs.index)
    for col in ["gauge_star", "gauge_end"]:
        attrs[col] = pd.to_datetime(attrs[col], format="%d/%m/%Y")
    return attrs


atributos_crudos = cargar_atributos(ids)
print(f"Atributos: {atributos_crudos.shape[0]} cuencas x {atributos_crudos.shape[1]} columnas")

Atributos: 248 cuencas x 75 columnas


### 1.4 Caudal faltante

Dentro del registro de cada estación, el caudal falta en el 6,1 % de los días. Casi todos son días sin medición (`no_data`); el resto son días anulados en el pre-filtrado por picos (`spike`) o valores congelados (`frozen`).

- **No se rellena.** El caudal es lo que el modelo aprende a predecir: rellenarlo haría que se entrenara y evaluara contra valores inventados. Los días faltantes quedan vacíos y NeuralHydrology los excluye de la pérdida y de las métricas. Sus forzantes siguen sirviendo como entrada para las ventanas vecinas.
- **No se descartan cuencas por tener vacíos.** El dataset ya exige 70 % de disponibilidad por estación. Un umbral de 5 % de vacíos eliminaría 136 de las 248 cuencas y reduciría la cobertura espacial.

En el código, el caudal se conserva solo en los días con estado `ok`; los demás quedan vacíos.

In [8]:
# El caudal se conserva solo en los días con estado "ok"; los demás quedan vacíos y no se imputan
for df in series.values():
    df.loc[df["streamflow_estado"] != "ok", "streamflow_m3s"] = np.nan

### 1.5 Forzantes: interpolación lineal de huecos de hasta 7 días

Las 4 forzantes (evapotraspiracion, temperatura maxima y minima, y precipitación) faltan los mismos días que el caudal. NeuralHydrology descarta toda ventana de entrenamiento con algún forzante vacío, y cada día vacío invalida las 365 ventanas que lo contienen: sin rellenar se perdería el 32 % de las ventanas.

- **Se interpolan solo los huecos internos de hasta 7 días** (con dato a ambos lados). Son el 83 % de los huecos, pero apenas el 10,5 % de los días vacíos (20.177 días).
- **Los huecos más largos quedan vacíos.** La interpolación lineal solo aproxima bien una variable que cambia poco dentro del hueco. Sobre varias semanas borraría la alternancia entre días secos (rachas de 2,7 días en mediana, 1.8) y lluviosos, que es la que explica los caudales bajos y altos. Además, ampliar el límite rinde cada vez menos: hasta 7 días se recuperan 17 ventanas por día interpolado; sin límite, menos de 1.
- **Los días rellenados quedan marcados** en la columna `forzantes_interpolados`, para revisar en la evaluación si los errores se concentran en ellos.

Las tablas de abajo muestran la duración de los huecos antes de interpolar y el efecto del límite sobre las ventanas de entrenamiento.

In [9]:
MAX_DIAS_INTERPOLACION = 7
forzantes = list(FORZANTES.values())


def rachas(evento):
    """Inicio y fin (exclusivo) de cada racha consecutiva de True."""
    bordes = np.flatnonzero(np.diff(np.r_[0, evento.astype(np.int8), 0]))
    return bordes[::2], bordes[1::2]


def interpolar_huecos_cortos(s, max_dias):
    """Interpola linealmente solo los huecos internos de hasta `max_dias` días.
    Devuelve la serie y una máscara con los días rellenados."""
    rellenar = np.zeros(len(s), dtype=bool)
    for ini, fin in zip(*rachas(s.isna().to_numpy())):
        if ini > 0 and fin < len(s) and fin - ini <= max_dias:
            rellenar[ini:fin] = True
    interpolada = s.interpolate(method="linear", limit_area="inside")
    return s.where(~rellenar, interpolada), rellenar


for df in series.values():
    if "forzantes_interpolados" in df:   # si la celda se vuelve a ejecutar, parte de los valores observados
        df.loc[df["forzantes_interpolados"], forzantes] = np.nan
    interpolado = np.zeros(len(df), dtype=bool)
    for col in forzantes:
        df[col], mascara = interpolar_huecos_cortos(df[col], MAX_DIAS_INTERPOLACION)
        interpolado |= mascara
    df["forzantes_interpolados"] = interpolado

print(f"Días con forzantes interpolados: {sum(int(df['forzantes_interpolados'].sum()) for df in series.values()):,}")

Días con forzantes interpolados: 20,177


In [10]:
# Duración de los huecos internos de las forzantes antes de interpolar (248 cuencas)
por_cuenca = {
    gid: ((df["pr"].isna() | df["forzantes_interpolados"]).to_numpy(),   # vacíos antes de interpolar
          df["streamflow_m3s"].notna().to_numpy(),
          EN_TRAIN)
    for gid, df in series.items()
}

duraciones = []
for vacio, _, _ in por_cuenca.values():
    ini, fin = rachas(vacio)
    interno = (ini > 0) & (fin < len(vacio))
    duraciones.extend((fin - ini)[interno])
duraciones = pd.Series(duraciones)

rangos = pd.cut(duraciones, [0, 1, 7, 30, 365, np.inf],
                labels=["1 día", "2 a 7 días", "8 a 30 días", "31 a 365 días", "más de 365 días"])
huecos = pd.DataFrame({"huecos": duraciones.groupby(rangos, observed=False).size(),
                       "días vacíos": duraciones.groupby(rangos, observed=False).sum()})
huecos["% de huecos"] = (100 * huecos["huecos"] / huecos["huecos"].sum()).round(1)
huecos["% de días vacíos"] = (100 * huecos["días vacíos"] / huecos["días vacíos"].sum()).round(1)
huecos

,huecos,días vacíos,% de huecos,% de días vacíos
1 día,7258,7258,53.2,3.8
2 a 7 días,4066,12919,29.8,6.7
8 a 30 días,1265,20326,9.3,10.6
31 a 365 días,972,101617,7.1,53.1
más de 365 días,74,49344,0.5,25.8


In [11]:
# Efecto del límite sobre las ventanas de entrenamiento: 365 días de forzantes completos
# que terminan en un día con caudal válido de 1981 a 2010
def evaluar_limite(limite):
    """Días interpolados y ventanas de entrenamiento válidas si se rellenan los huecos de hasta `limite` días."""
    dias_interpolados = ventanas = 0
    for vacio, q_valido, es_train in por_cuenca.values():
        ini, fin = rachas(vacio)
        rellenar = np.zeros(len(vacio), dtype=bool)
        for a, b in zip(ini, fin):
            if a > 0 and b < len(vacio) and b - a <= limite:
                rellenar[a:b] = True
        completo = (~vacio | rellenar).astype(int)
        ventana_completa = np.convolve(completo, np.ones(365, dtype=int))[: len(vacio)] == 365
        ventanas += int((ventana_completa & q_valido & es_train).sum())
        dias_interpolados += int(rellenar.sum())
    return dias_interpolados, ventanas

limites = {"sin interpolar": 0, "7 días": 7, "14 días": 14, "30 días": 30, "90 días": 90, "sin límite": np.inf}
compromiso = pd.DataFrame([evaluar_limite(l) for l in limites.values()], index=list(limites),
                          columns=["días interpolados", "ventanas de entrenamiento"])
compromiso["% de días vacíos interpolados"] = (100 * compromiso["días interpolados"] / duraciones.sum()).round(1)
compromiso["% de ventanas"] = (100 * compromiso["ventanas de entrenamiento"]
                               / compromiso["ventanas de entrenamiento"].max()).round(1)
compromiso["ventanas ganadas por día interpolado"] = (compromiso["ventanas de entrenamiento"].diff()
                                                      / compromiso["días interpolados"].diff()).round(1)
compromiso

,días interpolados,ventanas de entrenamiento,% de días vacíos interpolados,% de ventanas,ventanas ganadas por día interpolado
sin interpolar,0,1473247,0.0,67.6,NaN
7 días,20177,1819212,10.5,83.4,17.1
14 días,26975,1883762,14.1,86.4,9.5
30 días,40503,1976309,21.2,90.6,6.8
90 días,71761,2097075,37.5,96.2,3.9
sin límite,191464,2180431,100.0,100.0,0.7


### 1.6 Caudal en mm/día

El caudal viene en m³/s, una magnitud que crece con el tamaño de la cuenca. Se convierte a lámina de agua sobre la cuenca, la misma unidad de la precipitación y la evapotranspiración:

$$Q_{\text{mm/día}} = Q_{\text{m}^3\text{/s}} \times \frac{86{,}4}{\text{área}_{\text{km}^2}}$$

En mm/día la variación entre cuencas baja a un factor de 122 y el modelo aprende la respuesta de la cuenca, no su tamaño. Además, NeuralHydrology estandariza el caudal con una sola media y una sola desviación para todas las cuencas: en m³/s las dominarían los ríos grandes, y el caudal estandarizado de la cuenca mediana casi no variaría (desviación de 0,02, frente a 0,55 en mm/día). La media convertida coincide con el `q_mean` publicado en el dataset.

In [12]:
def caudal_a_mm(q_m3s, area_km2):
    """m³/s × 86 400 s/día × 1 000 mm/m ÷ (área × 10⁶ m²/km²) = m³/s × 86,4 / área."""
    return q_m3s * 86.4 / area_km2


for gid, df in series.items():
    df[OBJETIVO] = caudal_a_mm(df["streamflow_m3s"], atributos_crudos.at[gid, "area"])

In [13]:
# Caudal medio por cuenca en cada unidad (período de entrenamiento)
q = pd.DataFrame({gid: {"media_m3s": df.loc[EN_TRAIN, "streamflow_m3s"].mean(),
                        "media_mm": df.loc[EN_TRAIN, "streamflow"].mean(),
                        "desv_m3s": df.loc[EN_TRAIN, "streamflow_m3s"].std(),
                        "desv_mm": df.loc[EN_TRAIN, "streamflow"].std()}
                  for gid, df in series.items()}).T
desv_global_m3s, desv_global_mm = pd.concat([df.loc[EN_TRAIN, ["streamflow_m3s", "streamflow"]]
                                             for df in series.values()]).std()

media_total = pd.Series({gid: df["streamflow"].mean() for gid, df in series.items()})
razon = media_total / atributos_crudos["q_mean"]
print(f"Verificación: media convertida / q_mean publicado = {razon.median():.3f} (mediana entre cuencas)\n")

pd.DataFrame({
    "m³/s": [q["media_m3s"].min(), q["media_m3s"].max(), q["media_m3s"].max() / q["media_m3s"].min(),
             (q["desv_m3s"] / desv_global_m3s).median()],
    "mm/día": [q["media_mm"].min(), q["media_mm"].max(), q["media_mm"].max() / q["media_mm"].min(),
               (q["desv_mm"] / desv_global_mm).median()],
}, index=["caudal medio mínimo", "caudal medio máximo", "razón máximo / mínimo",
          "desviación del caudal estandarizado (cuenca mediana)"]).astype(float).round(3)

Verificación: media convertida / q_mean publicado = 1.000 (mediana entre cuencas)



,m³/s,mm/día
caudal medio mínimo,0.174,0.175
caudal medio máximo,7401.838,21.357
razón máximo / mínimo,42528.447,122.171
desviación del caudal estandarizado (cuenca mediana),0.024,0.551


### 1.7 Re-escalamiento y filtrado de atributos estáticos

- **Área.** Va de 25 a 260 372 km² y su distribución es muy asimétrica: estandarizada directamente, el 88 % de las cuencas quedaría entre −0,5 y +0,5 desviaciones y el modelo apenas distinguiría su tamaño. Entra como `log10(area)`, que reduce la asimetría de 5,9 a 0,5. El área original se conserva para convertir el caudal (1.6).
- **`tc_representative`.** Es un error del dataset: sus valores salen de las fórmulas de tiempo de concentración, pero están asignados a cuencas equivocadas. Solo 3 de 346 coinciden con una fórmula de su propia cuenca, y su correlación con la longitud del cauce es nula (−0,03, frente a 0,97 de `tc_Johnstone`). En la lista de candidatos (1.1) se reemplaza por `tc_Johnstone`, la fórmula de la que provienen más valores (211 de 346).
- **Atributos casi vacíos.** Un atributo de porcentaje (cobertura, geología, suelos o capacidad de uso) se descarta si menos del 5 % de las cuencas tiene más del 1 % de su área en esa clase. Sin contraste entre cuencas el modelo no puede aprender su efecto, y un valor distinto de cero solo identificaría a una cuenca. Se descartan `class_1`, `aridisols_perc` y `non_identi_land_perc`. Por el mismo criterio, `spodosols_perc`, `coal_mine_pit_perc` y `perpetual_snow_perc` ya se habían excluido de la lista de candidatos.

In [14]:
# Área: escala logarítmica
atributos = atributos_crudos.assign(log_area=np.log10(atributos_crudos["area"]))

# tc_representative no se usa: en la lista de candidatos (1.1) se reemplazó por tc_Johnstone

# Atributos de porcentaje casi vacíos
UMBRAL_PCT = 1.0          # % del área desde el que se considera que una clase está presente en la cuenca
MIN_FRAC_CUENCAS = 0.05   # fracción mínima de cuencas en las que debe estar presente
candidatos_pct = [c for g in ["cobertura", "geologia", "suelos", "capacidad_uso"] for c in ATRIBUTOS_CANDIDATOS[g]]
presencia = (atributos[candidatos_pct] > UMBRAL_PCT).mean()
descartados = presencia[presencia < MIN_FRAC_CUENCAS].index.tolist()
print("Atributos descartados por estar casi vacíos:", descartados)

Atributos descartados por estar casi vacíos: ['non_identi_land_perc', 'aridisols_perc', 'class_1']


In [15]:
# Área: distribución antes y después del logaritmo
area = atributos_crudos["area"]
z_area = (area - area.mean()) / area.std()
print(f"Área: {area.min():,.0f} a {area.max():,.0f} km² | asimetría {area.skew():.1f}, con log10 "
      f"{np.log10(area).skew():.1f} | sin log, {100 * (z_area.abs() < 0.5).mean():.0f} % de las cuencas "
      f"entre ±0,5 desviaciones")

# tc_representative: ¿coincide con alguna fórmula de su propia cuenca? (346 cuencas del dataset)
fisiograficos = leer_csv(DIR_DATOS / "10_CAMELS_COL_Physiograpic_characteristics.csv")
formulas_tc = ["tc_kirpich", "tc_chow", "tc_Johnstone", "tc_Engi_Corps"]
en_su_cuenca = (np.abs(fisiograficos[formulas_tc].to_numpy()
                       - fisiograficos[["tc_representative"]].to_numpy()) < 0.01).any(axis=1)

def formula_de_origen(valor):
    """Fórmula cuyo valor, en cualquier cuenca, coincide con `valor`."""
    coinciden = [f for f in formulas_tc if (np.abs(fisiograficos[f] - valor) < 0.01).any()]
    return coinciden[0] if len(coinciden) == 1 else ("ninguna" if not coinciden else "varias")

print(f"\ntc_representative coincide con una fórmula de su propia cuenca: {en_su_cuenca.sum()} de {len(fisiograficos)}")
print("Fórmula de la que proviene cada valor (en cualquier cuenca):",
      pd.Series([formula_de_origen(v) for v in fisiograficos["tc_representative"]]).value_counts().to_dict())
print(f"Spearman con la longitud del cauce: tc_representative "
      f"{atributos_crudos['tc_representative'].corr(atributos_crudos['streng_chanel'], method='spearman'):.2f}, "
      f"tc_Johnstone {atributos_crudos['tc_Johnstone'].corr(atributos_crudos['streng_chanel'], method='spearman'):.2f}")

# Atributos casi vacíos: cuencas con más de 1 % de su área en la clase (los 5 menos frecuentes)
print("\nCuencas con más de 1 % del área:", (atributos[candidatos_pct] > UMBRAL_PCT).sum().nsmallest(5).to_dict())

Área: 25 a 260,372 km² | asimetría 5.9, con log10 0.5 | sin log, 88 % de las cuencas entre ±0,5 desviaciones

tc_representative coincide con una fórmula de su propia cuenca: 3 de 346


Fórmula de la que proviene cada valor (en cualquier cuenca): {'tc_Johnstone': 211, 'tc_Engi_Corps': 73, 'varias': 57, 'tc_chow': 5}


Spearman con la longitud del cauce: tc_representative -0.03, tc_Johnstone 0.97

Cuencas con más de 1 % del área: {'class_1': 0, 'non_identi_land_perc': 1, 'aridisols_perc': 1, 'histosols_perc': 14, 'eroded_misce_perc': 16}


### 1.8 Índices climáticos estáticos recalculados

Los índices climáticos resumen el clima de cada cuenca en valores fijos:

| Índice | Definición |
|---|---|
| `p_mean`, `pet_mean` | Precipitación y evapotranspiración potencial medias (mm/día) |
| `aridity` | `pet_mean / p_mean` |
| `high_prec_freq`, `high_prec_dur` | Días por año con `pr ≥ 5 × p_mean`, y duración media de sus rachas |
| `low_prec_freq`, `low_prec_dur` | Días por año con `pr < 1 mm`, y duración media de sus rachas |

Los índices publicados en CAMELS-COL no se usan por dos razones. Se calcularon con todo el registro, hasta 2022, que incluye el período de prueba. Y usan la evapotranspiración anterior a su corrección (media de 8,9 frente a 3,3 mm/día), con la que la aridez mediana sería 1,62, propia de cuencas secas, en lugar de 0,59, propia de cuencas húmedas. Además, el `low_prec_dur` publicado (mediana de 16,4 días) no se pudo reproducir con su definición.

Por eso se recalculan con los forzantes observados de 1981 a 2010, sin días interpolados, y se agregan `p_mean` y `pet_mean`, que no vienen en el dataset. Los días vacíos cortan las rachas. Como solo usan forzantes, existen también en cuencas no aforadas.

In [16]:
def duracion_media(evento):
    """Duración media en días de las rachas consecutivas de True (0 si no hay)."""
    ini, fin = rachas(evento)
    return float((fin - ini).mean()) if len(ini) else 0.0


def indices_climaticos(pr, pet):
    """Índices climáticos con las definiciones de CAMELS-COL. Reciben solo valores
    observados; los días vacíos cortan las rachas."""
    anios = pr.notna().sum() / 365.25
    p_mean, pet_mean = pr.mean(), pet.mean()
    alta = (pr >= 5 * p_mean).to_numpy()   # NaN >= x es False: corta la racha
    seca = (pr < 1).to_numpy()
    return {"p_mean": p_mean, "pet_mean": pet_mean, "aridity": pet_mean / p_mean,
            "high_prec_freq": alta.sum() / anios, "high_prec_dur": duracion_media(alta),
            "low_prec_freq": seca.sum() / anios, "low_prec_dur": duracion_media(seca)}


climaticos = {}
for gid, df in series.items():
    train = df[EN_TRAIN]
    observado = ~train["forzantes_interpolados"]   # sin los días interpolados en 1.5
    climaticos[gid] = indices_climaticos(train["pr"].where(observado), train["poten_evapo"].where(observado))
atributos = atributos.assign(**pd.DataFrame.from_dict(climaticos, orient="index"))

In [17]:
# Índices publicados frente a recalculados: medianas entre cuencas y Spearman (orden de las cuencas)
originales = leer_csv(DIR_DATOS / "08_CAMELS_COL_Climatic_indices.csv").loc[ids]
indices = ["aridity", "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur"]
pd.DataFrame({
    "mediana original (1981–2022)": originales[indices].median(),
    "mediana recalculada (1981–2010)": atributos[indices].median(),
    "Spearman original vs recalculado": [atributos[c].corr(originales[c], method="spearman") for c in indices],
}).round(2)

,mediana original (1981–2022),mediana recalculada (1981–2010),Spearman original vs recalculado
aridity,1.62,0.59,1.00
high_prec_freq,9.67,9.84,0.99
high_prec_dur,1.36,1.04,0.88
low_prec_freq,157.94,163.79,0.99
low_prec_dur,16.43,2.65,0.87


### 1.9 Partición temporal

| Período | Años | Uso |
|---|---|---|
| Entrenamiento | 1981 a 2010 (30) | Ajuste de los pesos. Cubre varios ciclos de El Niño y La Niña |
| Validación | 2011 a 2015 (5) | Parada temprana y elección de `hidden_size` |
| Prueba | 2016 a 2022 (7) | Evaluación final, en los años más recientes (Yu & Tolson, 2026) |

La partición es cronológica, no aleatoria, porque el caudal diario está autocorrelacionado: con días al azar, cada día de prueba tendría a sus vecinos en el entrenamiento, y la métrica mediría la capacidad de interpolar entre días conocidos, no la de simular años no vistos.

No todas las cuencas tienen caudal en cada período: 248 en entrenamiento, 213 en validación y 189 en prueba. Las que no tienen datos de prueba se concentran en Caribe (39 %) y Amazonas (36 %), así que la evaluación temporal queda inclinada hacia Magdalena-Cauca; esto se declarará al reportar los resultados.

In [18]:
def asignar_periodo(fechas):
    """Etiqueta cada fecha con su período: train, validation o test."""
    etiqueta = np.full(len(fechas), "", dtype=object)
    for nombre, (ini, fin) in PERIODOS.items():
        etiqueta[(fechas >= ini) & (fechas <= fin)] = nombre
    return pd.Categorical(etiqueta, categories=list(PERIODOS))


for df in series.values():
    df["periodo"] = asignar_periodo(df.index)

In [19]:
# Cuencas con caudal en cada período y, por región, cuántas no tienen datos de prueba
dias_por_periodo = pd.DataFrame({
    "region": atributos["region"],
    **{f"dias_q_{p}": pd.Series({gid: int((df[OBJETIVO].notna() & (df["periodo"] == p)).sum())
                                 for gid, df in series.items()}) for p in PERIODOS},
})
print("Cuencas con caudal válido por período:",
      {p: int((dias_por_periodo[f"dias_q_{p}"] > 0).sum()) for p in PERIODOS}, "\n")

por_region = dias_por_periodo.groupby("region").agg(
    cuencas=("dias_q_test", "size"),
    con_caudal_en_test=("dias_q_test", lambda s: int((s > 0).sum())),
)
por_region.loc["Total"] = [len(dias_por_periodo), int((dias_por_periodo["dias_q_test"] > 0).sum())]
por_region = por_region.astype(int)
por_region["% sin caudal en test"] = (100 * (1 - por_region["con_caudal_en_test"] / por_region["cuencas"])).round(0)
por_region

Cuencas con caudal válido por período: {'train': 248, 'validation': 213, 'test': 189} 



,cuencas,con_caudal_en_test,% sin caudal en test
region,,,
Amazon,11,7,36.0
Caribbean,44,27,39.0
Magdalena-Cauca,136,108,21.0
Orinoco,37,31,16.0
Pacific,20,16,20.0
Total,248,189,24.0


### 1.10 Estandarización

El preprocesamiento no estandariza ninguna variable: los datos quedan en sus unidades físicas. NeuralHydrology calcula la media y la desviación de cada variable al entrenar, solo con las cuencas y el período de entrenamiento de esa corrida, y las guarda en `train_data/train_data_scaler.yml`.

Calcularlas con todo el dataset incluiría el período de prueba y las cuencas retenidas en la validación espacial, es decir, información de los datos con los que se evalúa. Además, en la validación espacial cada partición tiene otras cuencas de entrenamiento y los estadísticos deben recalcularse. Por eso este paso no tiene código propio.

### 1.11 Tabla final

Se verifica que los atributos del modelo existan y no tengan vacíos, y se unen las series de las 248 cuencas con sus atributos estáticos en una sola tabla, con una fila por cuenca y día. La base nueva se escribe en `data/processed/`:

| Archivo | Contenido |
|---|---|
| `camels_col_diario.parquet` | Tabla única: forzantes, caudal (m³/s y mm/día), estado del caudal, marca de interpolación, período y los 48 atributos estáticos |
| `atributos.csv` | Una fila por cuenca: atributos del modelo, auxiliares y firmas hidrológicas |
| `resumen_cuencas.csv` | Control de calidad por cuenca: caudal válido, días interpolados y días con caudal por período |

In [20]:
atributos_modelo = [c for cols in ATRIBUTOS_CANDIDATOS.values() for c in cols if c not in descartados]
faltan = [c for c in atributos_modelo if c not in atributos.columns]
assert not faltan, f"Atributos del modelo que no existen: {faltan}"
con_vacios = atributos[atributos_modelo].columns[atributos[atributos_modelo].isna().any()].tolist()
assert not con_vacios, f"Atributos del modelo con vacíos: {con_vacios}"

finales = {"forzantes": forzantes, "objetivo": OBJETIVO, "atributos_estaticos": atributos_modelo}

# Variables finales del modelo, por grupo
grupos = {"forzantes": forzantes,
          **{g: [c for c in cols if c not in descartados] for g, cols in ATRIBUTOS_CANDIDATOS.items()},
          "objetivo": [OBJETIVO]}
filas = [f"| {g} | {len(v)} | {', '.join(f'`{c}`' for c in v)} |" for g, v in grupos.items()]
display(Markdown("| Grupo | n | Variables |\n|---|---|---|\n" + "\n".join(filas)))

| Grupo | n | Variables |
|---|---|---|
| forzantes | 4 | `pr`, `poten_evapo`, `t_min`, `t_max` |
| fisiograficos | 10 | `log_area`, `gravelius_index`, `factor_form`, `maximum_ele`, `mean_ele`, `minimum_ele`, `streng_chanel`, `equi_slope`, `cn_catchment`, `tc_Johnstone` |
| climaticos | 7 | `p_mean`, `pet_mean`, `aridity`, `high_prec_freq`, `high_prec_dur`, `low_prec_freq`, `low_prec_dur` |
| cobertura | 5 | `forest_perc`, `nat_non_forest_form_perc`, `agricul_livestock_perc`, `non_vegeted_perc`, `water_bodies_perc_cob` |
| geologia | 6 | `unconso_depo_perc`, `hypabyssal_rock_perc`, `metamor_rock_perc`, `plutonic_rock_perc`, `sedimen_rock_perc`, `volcanic_rock_perc` |
| suelos | 13 | `inceptisols_perc`, `entisols_perc`, `andisols_perc`, `alfisols_perc`, `vertisols_perc`, `ultisols_perc`, `oxisols_perc`, `mollisols_perc`, `histosols_perc`, `urban_perc`, `eroded_misce_perc`, `rocky_misce_perc`, `water_bodies_perc_suelo` |
| capacidad_uso | 7 | `class_2`, `class_3`, `class_4`, `class_5`, `class_6`, `class_7`, `class_8` |
| objetivo | 1 | `streamflow` |

In [21]:
# Tabla única: series de todas las cuencas + atributos estáticos del modelo
columnas = ["date", "periodo", *forzantes, "streamflow_m3s", "streamflow_estado", "forzantes_interpolados", OBJETIVO]
diario = pd.concat([df.reset_index()[columnas].assign(gauge_id=gid) for gid, df in series.items()], ignore_index=True)
diario = diario[["gauge_id", *columnas]].merge(atributos[atributos_modelo].astype("float32"), left_on="gauge_id",
                                               right_index=True, how="left", validate="many_to_one")
flotantes = diario.select_dtypes("float64").columns
diario[flotantes] = diario[flotantes].astype("float32")
diario["gauge_id"] = diario["gauge_id"].astype("category")
diario["streamflow_estado"] = diario["streamflow_estado"].astype("category")

# Control de calidad por cuenca
en_registro = {gid: df["streamflow_estado"] != "sin_registro" for gid, df in series.items()}
resumen = pd.DataFrame([{
    "gauge_id": gid,
    "region": atributos.at[gid, "region"],
    "inicio_registro": df.index[en_registro[gid].to_numpy()].min().date(),
    "fin_registro": df.index[en_registro[gid].to_numpy()].max().date(),
    "q_valido_pct_registro": round(100 * df.loc[en_registro[gid], OBJETIVO].notna().mean(), 2),
    **{f"dias_q_{p}": int(dias_por_periodo.at[gid, f"dias_q_{p}"]) for p in PERIODOS},
    "dias_forzantes_interpolados": int(df["forzantes_interpolados"].sum()),
    "dias_forzantes_vacios_tras_interpolar": int(df.loc[en_registro[gid], "pr"].isna().sum()),
    "anios_forzantes_train": round(int((df.loc[EN_TRAIN, "pr"].notna()
                                        & ~df.loc[EN_TRAIN, "forzantes_interpolados"]).sum()) / 365.25, 1),
} for gid, df in series.items()])

DIR_SALIDA.mkdir(parents=True, exist_ok=True)
diario.to_parquet(DIR_SALIDA / "camels_col_diario.parquet", index=False)
atributos[AUXILIARES + atributos_modelo + FIRMAS_HIDROLOGICAS].rename_axis("gauge_id").to_csv(DIR_SALIDA / "atributos.csv")
resumen.to_csv(DIR_SALIDA / "resumen_cuencas.csv", index=False)

print(f"Tabla diaria: {len(diario):,} filas x {diario.shape[1]} columnas ({len(series)} cuencas x {len(CALENDARIO):,} días)")
print(f"Días con caudal válido: {int(diario[OBJETIVO].notna().sum()):,} | "
      f"días con forzantes interpolados: {int(diario['forzantes_interpolados'].sum()):,}")
print(f"Archivos escritos en {DIR_SALIDA.relative_to(ROOT)}/")

Tabla diaria: 3,804,320 filas x 59 columnas (248 cuencas x 15,340 días)
Días con caudal válido: 2,976,524 | días con forzantes interpolados: 20,177
Archivos escritos en data\processed/


### 1.12 Formato NeuralHydrology

NeuralHydrology no trae CAMELS-COL, así que se usa su lector genérico (`dataset: generic`), que exige esta estructura:

```
data/processed/
├── time_series/
│   ├── 11027010.nc            ← una serie por cuenca, coordenada `date`, vacíos como NaN
│   └── ...
└── attributes/
    └── atributos_modelo.csv   ← primera columna = id de la cuenca (se lee como texto)
```

- Los nombres de las variables en los `.nc` deben coincidir exactamente con `dynamic_inputs` y `target_variables` de la configuración.
- Solo se guardan las 4 forzantes y el caudal en mm/día.
- Los atributos no pueden tener vacíos (se verifica en 1.11).

In [22]:
# import sys
# !{sys.executable} -m pip install xarray

In [23]:
import xarray as xr

DIR_NH = DIR_SALIDA
(DIR_NH / "time_series").mkdir(parents=True, exist_ok=True)
(DIR_NH / "attributes").mkdir(parents=True, exist_ok=True)

variables_nc = finales["forzantes"] + [finales["objetivo"]]
for gid, g in diario.groupby("gauge_id", observed=True):
    xr.Dataset.from_dataframe(g.set_index("date")[variables_nc]).to_netcdf(DIR_NH / "time_series" / f"{gid}.nc")

atributos[finales["atributos_estaticos"]].rename_axis("gauge_id").to_csv(DIR_NH / "attributes" / "atributos_modelo.csv")

print(f"{len(list((DIR_NH / 'time_series').glob('*.nc')))} archivos .nc escritos\n")
with xr.open_dataset(DIR_NH / "time_series" / f"{ids[0]}.nc") as ds:
    print(ds)

248 archivos .nc escritos



<xarray.Dataset> Size: 430kB
Dimensions:      (date: 15340)
Coordinates:
  * date         (date) datetime64[ns] 123kB 1981-01-01 ... 2022-12-31
Data variables:
    pr           (date) float32 61kB ...
    poten_evapo  (date) float32 61kB ...
    t_min        (date) float32 61kB ...
    t_max        (date) float32 61kB ...
    streamflow   (date) float32 61kB ...
